# Imports

In [11]:
import kagglehub
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import itertools
import gc
import pandas as pd

import torch
import torch.nn as nn

from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import timm

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

seed=8359
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.backends.mps.is_available():
    torch.mps.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Globals

In [19]:
KAGGLE_DATASET_NAME="ciccionoss/rrdataset-split-10-10-80"
DATASET_PATH=os.path.join(os.getcwd(), "archive/")

MODEL_SAVE_PATH="/content/models/"
MODEL_NAME="vit_base_patch16_224"
HIDDEN_DIM = 256
DROPOUT = 0.3
ACTIVATION_NAME = "silu"

HR_LEARNING_RATES = [1e-4, 3e-5]
HR_WEIGHT_DECAYS = [1e-4, 1e-3]
HR_LAMBDA_TRANSFORMS = [0.1, 0.25, 0.5, 0.75, 1.0, 2.0]
HR_BATCH_SIZES = [32]
HR_NUM_EPOCHS = 10
HR_PATIENCE = 5

LEARNING_RATE = 3e-5
WEIGHT_DECAY = 1e-3
LAMBDA_TRANSFORM = 0.5
BATCH_SIZE = 32
NUM_EPOCHS = 20
PATIENCE = 7

NUM_WORKERS = 4
NUM_WORKERS_TEST = 2

In [13]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")

Device: mps


# Utils

In [14]:
class CustomDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.root = root
        self.samples = []

        real_ai_map = {
            "real": 0,
            "ai": 1
        }

        transformation_map = {
            "original": 0,
            "redigital": 1,
            "transfer": 2
        }

        for transform_type in os.listdir(root):
            transform_path = os.path.join(root, transform_type)

            if not os.path.isdir(transform_path):
                continue

            for class_type in os.listdir(transform_path):
                class_path = os.path.join(transform_path, class_type)

                if not os.path.isdir(class_path):
                    continue

                for image_name in os.listdir(class_path):
                    image_path = os.path.join(class_path, image_name)

                    if transform_type not in transformation_map:
                        continue

                    if class_type not in real_ai_map:
                        continue

                    transformation_label = transformation_map[transform_type]
                    real_ai_label = real_ai_map[class_type]

                    self.samples.append(
                        (image_path, real_ai_label, transformation_label)
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, real_ai_label, transformation_label = self.samples[idx]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, real_ai_label, transformation_label

    def __str__(self):
        return f"This dataset has {len(self.samples)} samples"

In [15]:
data_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [20]:
def create_dataloaders(train_data, val_data, test_data, batch_size, dvc, num_work):
    pin_memory = True if dvc == "cuda" else False

    train_loader = DataLoader(
        dataset=train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_work,
        pin_memory=pin_memory
    )

    val_loader = DataLoader(
        dataset=val_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_work,
        pin_memory=pin_memory
    )

    test_loader = DataLoader(
        dataset=test_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_work,
        pin_memory=pin_memory
    )

    return train_loader, val_loader, test_loader

# Data

In [10]:
print(f"Downloading {KAGGLE_DATASET_NAME} dataset!")
path = kagglehub.dataset_download(KAGGLE_DATASET_NAME)
print(f"Dataset downloaded at {path}!")

  1%|          | 100M/18.8G [00:23<1:14:13, 4.50MB/s] 


KeyboardInterrupt: 

In [21]:
path = DATASET_PATH

In [22]:
dataset_train = CustomDataset(
    root=path + "/train",
    transform=data_transform
)

dataset_val = CustomDataset(
    root=path + "/val",
    transform=data_transform
)

dataset_test = CustomDataset(
    root=path + "/test",
    transform=data_transform
)
print(f"Dataset train: {dataset_train}, Dataset val: {dataset_val}, Dataset test: {dataset_test}")

Dataset train: This dataset has 2550 samples, Dataset val: This dataset has 2549 samples, Dataset test: This dataset has 45900 samples


In [ ]:
# TODO: Verify the balancing of the dataset

# Network

In [13]:
class MultiHeadModel(nn.Module):
    def __init__(
        self,
        model_name="vit_base_patch16_224",
        num_transform_classes=3,
        hidden_dim=256,
        dropout=0.3,
        activation_name="gelu"
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        feature_dim = self.backbone.num_features

        match activation_name:
            case "gelu":
                activation_real_ai = nn.GELU()
                activation_transform = nn.GELU()
            case "relu":
                activation_real_ai = nn.ReLU()
                activation_transform = nn.ReLU()
            case "leaky_relu":
                activation_real_ai = nn.LeakyReLU(0.01)
                activation_transform = nn.LeakyReLU(0.01)
            case "silu":
                activation_real_ai = nn.SiLU()
                activation_transform = nn.SiLU()
            case _:
                activation_real_ai = nn.GELU()
                activation_transform = nn.GELU()

        self.real_ai_head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            activation_real_ai,
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

        self.transformation_head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, hidden_dim),
            activation_transform,
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_transform_classes)
        )

    def forward(self, x):
        features = self.backbone(x)

        real_ai_logits = self.real_ai_head(features).squeeze(1)
        transformation_logits = self.transformation_head(features)

        return real_ai_logits, transformation_logits

# Train

In [14]:
def evaluate_model(model, data_loader, lambda_transform, device):
    model.eval()

    bce_loss = nn.BCEWithLogitsLoss()
    ce_loss = nn.CrossEntropyLoss()

    total_loss = 0
    total_real_ai_loss = 0
    total_transform_loss = 0
    total_samples = 0

    all_fake_labels = []
    all_fake_preds = []

    all_transform_labels = []
    all_transform_preds = []

    with torch.no_grad():
        for images, fake_labels, transform_labels in tqdm(
            data_loader,
            desc="Validation"
          ):
            images = images.to(device)
            fake_labels = fake_labels.to(device)
            transform_labels = transform_labels.to(device)

            fake_logits, transform_logits = model(images)

            loss_fake = bce_loss(fake_logits, fake_labels.float())
            loss_transform = ce_loss(transform_logits, transform_labels)

            loss = loss_fake + lambda_transform * loss_transform

            batch_size = images.size(0)

            total_loss += loss.item() * batch_size
            total_real_ai_loss += loss_fake.item() * batch_size
            total_transform_loss += loss_transform.item() * batch_size
            total_samples += batch_size

            fake_preds = (torch.sigmoid(fake_logits) > 0.5).long()
            transform_preds = torch.argmax(transform_logits, dim=1)

            all_fake_labels.extend(fake_labels.cpu().numpy())
            all_fake_preds.extend(fake_preds.cpu().numpy())

            all_transform_labels.extend(transform_labels.cpu().numpy())
            all_transform_preds.extend(transform_preds.cpu().numpy())

    avg_loss = total_loss / total_samples
    avg_real_ai_loss = total_real_ai_loss / total_samples
    avg_transform_loss = total_transform_loss / total_samples

    real_ai_acc = accuracy_score(all_fake_labels, all_fake_preds)
    real_ai_precision = precision_score(
        all_fake_labels,
        all_fake_preds,
        zero_division=0
    )
    real_ai_recall = recall_score(
        all_fake_labels,
        all_fake_preds,
        zero_division=0
    )
    real_ai_f1 = f1_score(
        all_fake_labels,
        all_fake_preds,
        zero_division=0
    )

    transform_acc = accuracy_score(
        all_transform_labels,
        all_transform_preds
    )
    transform_macro_f1 = f1_score(
        all_transform_labels,
        all_transform_preds,
        average="macro",
        zero_division=0
    )

    real_ai_confusion_matrix = confusion_matrix(
        all_fake_labels,
        all_fake_preds,
        labels=[0, 1]
    ).tolist()
    transform_confusion_matrix = confusion_matrix(
        all_transform_labels,
        all_transform_preds,
        labels=[0, 1, 2]
    ).tolist()

    return {
        "loss": avg_loss,
        "real_ai_loss": avg_real_ai_loss,
        "transform_loss": avg_transform_loss,

        "real_ai_acc": real_ai_acc,
        "real_ai_precision": real_ai_precision,
        "real_ai_recall": real_ai_recall,
        "real_ai_f1": real_ai_f1,

        "transform_acc": transform_acc,
        "transform_macro_f1": transform_macro_f1,

        "real_ai_confusion_matrix": real_ai_confusion_matrix,
        "transform_confusion_matrix": transform_confusion_matrix
    }

In [15]:
def train_model(
    model_name,
    learning_rate,
    weight_decay,
    lambda_transform,
    batch_size,
    hidden_dim,
    dropout,
    activation_name,
    num_epochs=10,
    patience=3
):

  train_loader, val_loader, _ = create_dataloaders(
    dataset_train,
    dataset_val,
    dataset_test,
    batch_size,
    device,
    NUM_WORKERS
  )

  model = MultiHeadModel(
      model_name=model_name,
      num_transform_classes=3,
      hidden_dim=hidden_dim,
      dropout=dropout,
      activation_name=activation_name
  ).to(device)

  bce_loss = nn.BCEWithLogitsLoss()
  ce_loss = nn.CrossEntropyLoss()

  optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
  )

  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=learning_rate*0.01
  )

  history = []

  best_score=-1.0
  best_epoch=0
  best_val_metrics = None
  epochs_without_improvements=0

  model_file_name = (
      f"{model_name}_"
    + f"lr-{learning_rate}_"
    + f"wd-{weight_decay}_"
    + f"lt-{lambda_transform}_"
    + f"bs-{batch_size}_"
    + f"hd-{hidden_dim}_"
    + f"d-{dropout}_"
    + f"a-{activation_name}_"
    + f"seed-{SEED}_MultiHead.pth"
  )

  if not os.path.exists(MODEL_SAVE_PATH):
    os.makedirs(MODEL_SAVE_PATH)
  model_path = os.path.join(MODEL_SAVE_PATH, model_file_name)

  for epoch in range(num_epochs):
    model.train()

    current_lr = optimizer.param_groups[0]["lr"]

    train_total_loss = 0.0
    train_real_ai_loss = 0.0
    train_transform_loss = 0.0
    train_total_samples = 0

    all_train_fake_labels = []
    all_train_fake_preds = []

    all_train_transform_labels = []
    all_train_transform_preds = []

    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")

    for images, fake_labels, transform_labels in tqdm(
        train_loader,
        desc=(
            f"Train | LR={learning_rate} | WD={weight_decay} | "
            f"LT={lambda_transform} | BS={batch_size} | "
            f"Epoch {epoch + 1}/{num_epochs}"
        )
    ):
      images = images.to(device)

      fake_labels = fake_labels.to(device)
      transform_labels = transform_labels.to(device)

      fake_logits, transform_logits = model(images)

      loss_fake = bce_loss(fake_logits, fake_labels.float())
      loss_transform = ce_loss(transform_logits, transform_labels)

      loss = loss_fake + lambda_transform * loss_transform

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      batch_size_current = images.size(0)

      train_total_loss += loss.item() * batch_size_current
      train_real_ai_loss += loss_fake.item() * batch_size_current
      train_transform_loss += loss_transform.item() * batch_size_current
      train_total_samples += batch_size_current

      fake_preds = (torch.sigmoid(fake_logits) > 0.5).long()
      transform_preds = torch.argmax(transform_logits, dim=1)

      all_train_fake_labels.extend(fake_labels.cpu().numpy())
      all_train_fake_preds.extend(fake_preds.cpu().numpy())

      all_train_transform_labels.extend(transform_labels.cpu().numpy())
      all_train_transform_preds.extend(transform_preds.cpu().numpy())

    avg_train_loss = train_total_loss / train_total_samples
    avg_train_real_ai_loss = train_real_ai_loss / train_total_samples
    avg_train_transform_loss = train_transform_loss / train_total_samples

    train_real_ai_acc = accuracy_score(
        all_train_fake_labels,
        all_train_fake_preds
    )
    train_real_ai_f1 = f1_score(
        all_train_fake_labels,
        all_train_fake_preds,
        zero_division=0
    )
    train_real_ai_confusion_matrix = confusion_matrix(
        all_train_fake_labels,
        all_train_fake_preds,
        labels=[0, 1]
    ).tolist()
    train_transform_acc = accuracy_score(
        all_train_transform_labels,
        all_train_transform_preds
    )
    train_transform_macro_f1 = f1_score(
        all_train_transform_labels,
        all_train_transform_preds,
        average="macro",
        zero_division=0
    )
    train_transform_confusion_matrix = confusion_matrix(
        all_train_transform_labels,
        all_train_transform_preds,
        labels=[0, 1, 2]
    ).tolist()

    print("-" * 80)
    print(
      f"LR={learning_rate} | Current LR={current_lr:.2e} | "
      f"WD={weight_decay} | Lambda={lambda_transform} | BS={batch_size}"
    )
    print(
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Train Real AI Loss: {avg_train_real_ai_loss:.4f} | "
        f"Train Transform Loss: {avg_train_transform_loss:.4f}\n"
        f"Train Real AI Acc: {train_real_ai_acc:.4f} | "
        f"Train Real AI F1: {train_real_ai_f1:.4f} \n"
        f"Train Transform Acc: {train_transform_acc:.4f} | "
        f"Train Transform Macro F1: {train_transform_macro_f1:.4f} |"
        f"Train Real AI Confusion Matrix: \n{train_real_ai_confusion_matrix}\n"
        f"Train Transform Confusion Matrix: \n{train_transform_confusion_matrix}"
    )
    print()
    print("Starting Validation!")
    print("-" * 80)

    val_metrics = evaluate_model(
        model,
        val_loader,
        lambda_transform,
        device
    )

    scheduler.step()

    epoch_result = {
        "epoch": epoch + 1,
        "learning_rate": learning_rate,
        "current_lr": current_lr,
        "weight_decay": weight_decay,
        "lambda_transform": lambda_transform,
        "batch_size": batch_size,
        "hidden_dim": hidden_dim,
        "dropout": dropout,
        "activation": activation_name,

        "train_loss": avg_train_loss,
        "train_real_ai_loss": avg_train_real_ai_loss,
        "train_transform_loss": avg_train_transform_loss,
        "train_real_ai_acc": train_real_ai_acc,
        "train_real_ai_f1": train_real_ai_f1,
        "train_transform_acc": train_transform_acc,
        "train_transform_macro_f1": train_transform_macro_f1,
        "train_real_ai_confusion_matrix": train_real_ai_confusion_matrix,
        "train_transform_confusion_matrix": train_transform_confusion_matrix,

        "val_loss": val_metrics["loss"],
        "val_real_ai_loss": val_metrics["real_ai_loss"],
        "val_transform_loss": val_metrics["transform_loss"],
        "val_real_ai_acc": val_metrics["real_ai_acc"],
        "val_real_ai_precision": val_metrics["real_ai_precision"],
        "val_real_ai_recall": val_metrics["real_ai_recall"],
        "val_real_ai_f1": val_metrics["real_ai_f1"],
        "val_real_ai_confusion_matrix": val_metrics["real_ai_confusion_matrix"],
        "val_transform_acc": val_metrics["transform_acc"],
        "val_transform_macro_f1": val_metrics["transform_macro_f1"],
        "val_transform_confusion_matrix": val_metrics["transform_confusion_matrix"]
    }

    history.append(epoch_result)

    print(
        f"Validation Loss: {val_metrics["loss"]:.4f} | "
        f"Validation Real AI Loss: {val_metrics["real_ai_loss"]:.4f} | "
        f"Validation Transform Loss: {val_metrics["transform_loss"]:.4f}\n"
        f"Validation Real AI Acc: {val_metrics["real_ai_acc"]:.4f} | "
        f"Validation Real AI F1: {val_metrics["real_ai_f1"]:.4f}\n"
        f"Validation Transform Acc: {val_metrics["transform_acc"]:.4f} | "
        f"Validation Transform Macro F1: {val_metrics["transform_macro_f1"]:.4f}\n"
        f"Validation Real AI Confusion Matrix: \n{val_metrics["real_ai_confusion_matrix"]}\n"
        f"Validation Transform Confusion Matrix: \n{val_metrics["transform_confusion_matrix"]}"
    )
    print("-" * 80 + "\n")

    current_score = (
      0.8 * val_metrics["real_ai_acc"]
      + 0.2 * val_metrics["transform_acc"]
    )

    if current_score > best_score:
      best_score = current_score
      best_epoch = epoch + 1
      best_val_metrics = val_metrics
      epochs_without_improvements = 0
      torch.save(
          model.state_dict(),
          model_path
      )
      print (f"Saved new best model: {model_path}!")
    else:
      epochs_without_improvements += 1

    if epochs_without_improvements >= patience:
      print(
        f"Early stopping at epoch {epoch + 1}! |"
        f"No improvements in {patience} epochs."
      )
      break
  best_result = {
    "best_epoch": best_epoch,
    "best_score": best_score,
    "best_val_metrics": best_val_metrics,
    "history": history
  }

  del model
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
  return best_result

With Hyper-Parameter Research

Without Hyper-Parameter Research

In [ ]:
best_result_no_hr = train_model(
    model_name=MODEL_NAME,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lambda_transform=LAMBDA_TRANSFORM,
    batch_size=BATCH_SIZE,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    activation_name=ACTIVATION_NAME,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Train | LR=3e-05 | WD=0.001 | LT=0.5 | BS=32 | Epoch 1/20: 100%|██████████| 160/160 [04:48<00:00,  1.80s/it]


--------------------------------------------------------------------------------
Epoch [1/20]
LR=3e-05 | Current LR=3.00e-05 | WD=0.001 | Lambda=0.5 | BS=32
Train Loss: 0.6642 | Train Real AI Loss: 0.3176 | Train Transform Loss: 0.6932 |Train Real AI Acc: 0.8688 | Train Real AI F1: 0.8685Train Transform Acc: 0.6864 | Train Transform Macro F1: 0.6865 |Train Real AI Confusion Matrix: 
[[2220, 329], [340, 2210]] |Train Transform Confusion Matrix: 
[[1113, 217, 370], [232, 1325, 142], [471, 167, 1062]]

Starting Validation!
--------------------------------------------------------------------------------


Validation: 100%|██████████| 160/160 [03:08<00:00,  1.18s/it]


...

Train | LR=3e-05 | WD=0.001 | LT=0.5 | BS=32 | Epoch 8/20: 100%|██████████| 160/160 [04:49<00:00,  1.81s/it]


--------------------------------------------------------------------------------
Epoch [8/20]
LR=3e-05 | Current LR=2.19e-05 | WD=0.001 | Lambda=0.5 | BS=32
Train Loss: 0.0184 | Train Real AI Loss: 0.0009 | Train Transform Loss: 0.0350 |Train Real AI Acc: 1.0000 | Train Real AI F1: 1.0000Train Transform Acc: 0.9882 | Train Transform Macro F1: 0.9882 |Train Real AI Confusion Matrix: 
[[2549, 0], [0, 2550]] |Train Transform Confusion Matrix: 
[[1669, 1, 30], [0, 1696, 3], [22, 4, 1674]]

Starting Validation!
--------------------------------------------------------------------------------


Validation: 100%|██████████| 160/160 [02:47<00:00,  1.04s/it]


Validation Loss: 0.4233 | Validation Real AI Loss: 0.2327 | Validation Transform Loss: 0.3812 |Validation Real AI Acc: 0.9416 | Validation Real AI F1: 0.9416Validation Transform Acc: 0.8806 | Validation Transform Macro F1: 0.8808 |Validation Real AI Confusion Matrix: 
[[2399, 151], [147, 2403]] |Validation Transform Confusion Matrix: 
[[1345, 30, 325], [58, 1605, 37], [153, 6, 1541]]
--------------------------------------------------------------------------------



Train | LR=3e-05 | WD=0.001 | LT=0.5 | BS=32 | Epoch 9/20:  11%|█         | 17/160 [00:31<04:05,  1.71s/it]